In [ ]:
dev = "xxx"  # key hidden

1. Function to scrape youtube comments from a given Video ID

In [2]:
import googleapiclient.discovery
import pandas as pd

api_service_name = "youtube"
api_version = "v3"
DEVELOPER_KEY = dev

youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)


def getcomments(video):
  request = youtube.commentThreads().list(
      part="snippet",
      videoId=video,
      maxResults=100
  )

  comments = []

  # Execute the request.
  response = request.execute()

  # Get the comments from the response.
  for item in response['items']:
      comment = item['snippet']['topLevelComment']['snippet']
      public = item['snippet']['isPublic']
      comments.append([
          comment['authorDisplayName'],
          comment['publishedAt'],
          comment['likeCount'],
          comment['textOriginal'],
          comment['videoId'],
          public
      ])

  while (1 == 1):
    try:
     nextPageToken = response['nextPageToken']
    except KeyError:
     break
    nextPageToken = response['nextPageToken']
    # Create a new request object with the next page token.
    nextRequest = youtube.commentThreads().list(part="snippet", videoId=video, maxResults=100, pageToken=nextPageToken)
    # Execute the next request.
    response = nextRequest.execute()
    # Get the comments from the next response.
    for item in response['items']:
      comment = item['snippet']['topLevelComment']['snippet']
      public = item['snippet']['isPublic']
      comments.append([
          comment['authorDisplayName'],
          comment['publishedAt'],
          comment['likeCount'],
          comment['textOriginal'],
          comment['videoId'],
          public
      ])

  df2 = pd.DataFrame(comments, columns=['author', 'updated_at', 'like_count', 'text','video_id','public'])
  return df2

2. Scrape the Video ID's of top n Youtube videos about a given Topic (Most Relevant)

In [3]:
youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)

def get_top_video_ids(topic, max_results=5):
    request = youtube.search().list(
        part="snippet",
        q=topic,
        type="video",
        maxResults=max_results,
        order="relevance"
    )

    response = request.execute()

    video_ids = []

    for item in response["items"]:
        video_ids.append(item["id"]["videoId"])

    return video_ids

In [4]:
topics = [
    "interstellar ending explained",
    "interstellar theory",
    "interstellar explained",
    "interstellar movie analysis",
    "interstellar ending breakdown"
]

all_video_ids = []

for topic in topics:

    print("Searching:", topic)

    video_ids = get_top_video_ids(topic, 15)

    all_video_ids.extend(video_ids)


# remove duplicate video IDs
all_video_ids = list(set(all_video_ids))

print("Total unique videos:", len(all_video_ids))

all_video_ids

Searching: interstellar ending explained
Searching: interstellar theory
Searching: interstellar explained
Searching: interstellar movie analysis
Searching: interstellar ending breakdown
Total unique videos: 40


['4f9V-8BHONo',
 'CQpTKQloHuI',
 'iWrFVvEgFYI',
 'I1AXx1XmQrc',
 '5I8YSiSp8rc',
 'vVRrEBgNKK4',
 '9YBpOBIsZoc',
 '2mUSh5N-gIY',
 'qhW1HfSuPVQ',
 'bU-pwPhbOTw',
 'X90Cxi3Uwrg',
 'HdJD2TIlAQY',
 'XD8ycTPJov8',
 'BHsFzDON6pA',
 'qOngRXgUTN4',
 'Ibcs3bVtVR8',
 'QBSw8nSVpmI',
 'VSuF_hK3qMI',
 'D09sle7hgWw',
 'zBNKrja8dyY',
 'j3DuONZb3Ik',
 'NC78S8jNy4g',
 'JqKa6qyVYgg',
 'IIo5Uo-gxJI',
 'R1cexcjdyIE',
 'UvmqsjHdXRU',
 't6kqaip7WS4',
 'dMl_GhbcP0c',
 'Et9cKWWZ0LM',
 'kcVdY-PKHdk',
 '1w_DVgaDQ_g',
 'JZ9qjB2PVuE',
 'ASkz5prADQA',
 'Res32GunId8',
 'XiDFc5aIqFg',
 'NXhdt1R6TYE',
 'bpbK6LZ4lQ0',
 'yVPpMMMxl0Y',
 '4-6zBotsi-s',
 'IZ-Jzl_57cU']

In [5]:
all_dfs = []

for video in all_video_ids:
    df = getcomments(video)
    all_dfs.append(df)

comments_df = pd.concat(all_dfs, ignore_index=True)
comments_df.head()

,author,updated_at,like_count,text,video_id,public
0,@StarTalk,2024-12-03T20:51:01Z,407,Go to https://ground.news/startalk to stay ful...,4f9V-8BHONo,True
1,@jacobfoulds7618,2026-05-19T04:31:34Z,1,Aaaaaaahhhereee,4f9V-8BHONo,True
2,@djangoknight,2026-05-18T06:58:35Z,1,U📈hhhh,4f9V-8BHONo,True
3,@sarge505050,2026-05-16T22:18:49Z,0,Kip Thorne gave me some ideas piezoelectric..,4f9V-8BHONo,True
4,@tristananastasio9486,2026-05-16T05:10:48Z,0,Everyone’s always making Kip “do a calculation” 🫠,4f9V-8BHONo,True


In [6]:
comments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54427 entries, 0 to 54426
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   author      54427 non-null  object
 1   updated_at  54427 non-null  object
 2   like_count  54427 non-null  object
 3   text        54427 non-null  object
 4   video_id    54427 non-null  object
 5   public      54427 non-null  object
dtypes: object(6)
memory usage: 2.5+ MB


Changing the order of the comments according to the Likes

In [7]:
comments_df = comments_df.sort_values(
    by="like_count",
    ascending=False
)

In [8]:
pd.set_option('display.max_colwidth', None)

comments_df.sort_values(
    by="like_count",
    ascending=False
)[["like_count", "text"]].head(20)

,like_count,text
27655,41736,The guy who stay in the ship for 20 years.\n\nHe's the king of quarantine.
34522,37032,"A husband waiting for his wife shopping feels a very long time, while she feel only few minutes. A real time dilation."
24858,17975,My wife keeps asking me to dust my office. I'm like - what if my dad wants to talk to me?
28973,15427,"The scene where cooper comes back to the ship after the ocean planet incident , and sees all the video transmissions from his children gets me every time."
37552,15359,I was disappointed to see that Christopher Nolan did not add a disclaimer to the movie warning people not to go into black holes. Now everyone's going to be doing it.
29106,14772,Did you also find Interstellar confusing?
20126,14772,Did you also find Interstellar confusing?
26807,14460,This movie was made waaaaay too early for its time.
4451,13627,"This shit was filmed in my hometown. If I recall correctly production gave all the fucking crops they actually had to grow for the opening scenes of the film to local farmers to boost the local economy, and I'll always respect how much they helped with that."
19555,10878,You know it's a legendary film when you can make a breakdown of it almost 9 years after its release and people are still thirsty for it.


Convert the Dataframe to CSV file as raw data

In [9]:
comments_df.to_csv(
    "raw_data.csv",
    index=False
)